In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import xml.etree.ElementTree as ET
from datetime import datetime

In [0]:
CATALOG         = "bfsi"
SCHEMA          = "bronze_layer"
SOURCE_PATH     = "/Volumes/bfsi/landing/neft_rtgs/neft_rtgs_transactions.xml"  # for header extraction
SOURCE_DIR      = "/Volumes/bfsi/landing/neft_rtgs/"  # for Auto Loader
TABLE_NAME      = f"{CATALOG}.{SCHEMA}.neft_rtgs_transactions"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/neft_rtgs/_checkpoint/{SCHEMA}_neft_rtgs_transactions"
SOURCE_SYSTEM   = "NEFT_RTGS"
BATCH_ID        = dbutils.widgets.get("batch_id") if "batch_id" in [w.name for w in dbutils.widgets.getAll()] else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

In [0]:
# Read the XML file content
xml_content = dbutils.fs.head(SOURCE_PATH, 10000000)  # Read up to 10MB
tree = ET.fromstring(xml_content)

# Extract header info
grp_hdr = tree.find(".//GrpHdr")
msg_id = grp_hdr.find("MsgId").text
nb_of_txs = grp_hdr.find("NbOfTxs").text
total_amt_element = grp_hdr.find("TtlIntrBkSttlmAmt")
total_settlement_amt = float(total_amt_element.text)
total_settlement_ccy = total_amt_element.get("Ccy", "INR")

print(f"Message ID: {msg_id}")
print(f"Number of Transactions: {nb_of_txs}")
print(f"Total Settlement Amount: {total_settlement_ccy} {total_settlement_amt:,.2f}")


In [0]:
# Define schema matching Spark XML reader output for CdtTrfTxInf elements
neft_xml_schema = StructType([
    StructField("InstrId", StringType(), True),
    StructField("Amt", StructType([
        StructField("_VALUE", DoubleType(), True),
        StructField("_Ccy", StringType(), True),
    ]), True),
    StructField("Dbtr", StructType([
        StructField("Nm", StringType(), True),
    ]), True),
    StructField("Cdtr", StructType([
        StructField("Nm", StringType(), True),
    ]), True),
    StructField("Purp", StructType([
        StructField("Cd", StringType(), True),
    ]), True),
    StructField("RmtInf", StructType([
        StructField("Ustrd", StringType(), True),
    ]), True),
])

# Auto Loader: incremental ingestion of XML files
df_neft_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "xml")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("rowTag", "CdtTrfTxInf")
    .schema(neft_xml_schema)
    .load(SOURCE_DIR)
)

print("Auto Loader stream configured for NEFT/RTGS XML transactions")

In [0]:
# Flatten nested XML structs and add metadata columns
df_neft_bronze = (
    df_neft_stream
    .select(
        F.col("InstrId").alias("instr_id"),
        F.col("Amt._VALUE").alias("amount"),
        F.col("Amt._Ccy").alias("currency"),
        F.col("Dbtr.Nm").alias("debtor_name"),
        F.col("Cdtr.Nm").alias("creditor_name"),
        F.col("Purp.Cd").alias("purpose_code"),
        F.col("RmtInf.Ustrd").alias("remittance_info"),
        # Preserve raw parsed XML struct as JSON for programmatic access
        F.to_json(
            F.struct("InstrId", "Amt", "Dbtr", "Cdtr", "Purp", "RmtInf")
        ).alias("_raw_xml_content"),
    )
    .withColumn("msg_id", F.lit(msg_id))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id", F.lit(f"BATCH_NEFT_{msg_id}"))
    .withColumn("_total_settlement_amt", F.lit(total_settlement_amt))
)

# Write stream to Delta table in bronze_layer
query = (
    df_neft_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)

query.awaitTermination()
print(f"Auto Loader stream completed for {TABLE_NAME}")

In [0]:
# Verify data in bronze Delta table
df_verify = spark.table(TABLE_NAME)
print(f"Total NEFT/RTGS transactions ingested: {df_verify.count()}")
display(df_verify.limit(5))

# Preserve variables for downstream use
neft_total_settlement_amt = total_settlement_amt
neft_nb_of_txs = nb_of_txs